In [8]:
"""
ReAct-RAG -- Reason + Act Retrieval-Augmented Generation
=========================================================
Primary Reference : Yao et al., "ReAct: Synergizing Reasoning and Acting
                   in Language Models"
                   arXiv: 2210.03629  (ICLR 2023, Princeton + Google Brain)
                   Project: https://react-lm.github.io/

Architecture: FIVE configurations covering the complete ReAct-RAG design space:
    (1) CoT-Only Baseline       -- chain-of-thought reasoning, NO tool calls,
                                   no external retrieval. Pure parametric.
    (2) Act-Only Baseline       -- tool calls only, NO explicit reasoning traces.
                                   Actions without Thoughts.
    (3) Core ReAct              -- canonical Thought -> Action -> Observation
                                   loop with a multi-tool registry:
                                   Search[query], Lookup[term], Calculate[expr],
                                   Verify[claim], Finish[answer]
    (4) ReAct + CoT-SC          -- ReAct with Chain-of-Thought Self-Consistency:
                                   sample N=3 trajectories (T=0.7), select by
                                   majority vote on final answers
    (5) ReAct + Reflection      -- full production: ReAct loop with failure
        [BEST]                     detection and self-reflection. On retrieval
                                   failure or reasoning loop: generate a
                                   reflection critique and retry with corrected
                                   strategy. Full trajectory audit.

=============================================================================
THE FUNDAMENTAL INSIGHT: WHY THOUGHT AND ACTION CANNOT BE SEPARATED
=============================================================================
Prior to ReAct, reasoning and acting were studied as separate capabilities:
    Chain-of-Thought (Wei et al., 2022): reasoning only.
        The model generates internal reasoning traces ("Thought 1 ... Thought 2 ...")
        but NEVER interacts with external sources. All knowledge comes from
        parametric memory. Result: fact hallucination when parametric memory
        is wrong or outdated. The model invents facts rather than retrieve them.
        Example failure: "Mystere is performed at the Bellagio Hotel."
        (Wrong. It's the Treasure Island Hotel -- the model hallucinated.)

    Acting-only prompting (WebGPT, SayCan): actions without reasoning.
        The model selects and executes tool calls but does NOT generate explicit
        reasoning traces about WHY a particular action is needed.
        Result: inability to synthesize across multiple retrieved observations.
        The model can retrieve facts but cannot reason about them in combination.
        Example failure: retrieves "Mystere is performed at Treasure Island."
        retrieves "Treasure Island has 2884 rooms."
        But cannot synthesize: "Therefore the answer is 2884."

ReAct's insight: <b>reasoning traces help the model TARGET what to retrieve next,
and retrieved observations GROUND the reasoning in verifiable facts.</b>

The synergy is bidirectional:
    Thought 1: "I need to find where Mystere is performed to get the hotel."
                -> This thought TARGETS Action 1: Search[Mystere show Las Vegas]
    Observation 1: "Mystere is performed at Treasure Island Hotel and Casino."
                -> This observation UPDATES Thought 2: "Now I need the room count."
    Thought 2: "I found the hotel. Now I need to find its number of rooms."
                -> This updated thought TARGETS Action 2: Search[Treasure Island rooms]
    Observation 2: "The hotel has 2,884 rooms."
                -> This observation ENABLES Thought 3 + Finish[2884]

Neither CoT alone nor Act alone can complete this trajectory. The SYNERGY of
alternating Thought+Action+Observation is the contribution of ReAct.

=============================================================================
FORMAL DEFINITION: THE REACT AUGMENTED ACTION SPACE
=============================================================================
Standard RL/agent formulation:
    At each timestep t: the agent receives observation o_t in O,
    selects action a_t in A, transitions to next state.

ReAct augments the action space: B = A UNION L
    A: actions that INTERACT WITH THE ENVIRONMENT (tool calls):
       Search[query], Lookup[term], Calculate[expr], Verify[claim], Finish[answer]
    L: language space for THOUGHTS (internal reasoning):
       Thought: "I need to find X because Y."

Thoughts in L do NOT trigger environment state transitions:
    a_t in L -> observation o_t+1 is the same as o_t (no side effects).
Thoughts only UPDATE the CONTEXT c_t = (o_1, a_1, ..., o_t-1, a_t-1):
    by adding a Thought token, the context is enriched for the next step.

Policy: pi(a_t | c_t) implemented by frozen LLM prompted with few-shot examples.
The LLM generates the NEXT TOKEN autoregressively whether it is part of a
Thought or an Action -- there is no explicit switching mechanism.
The format "Thought: ..." or "Action: ..." in the context guides the LLM
to generate the appropriate content via in-context learning.

=============================================================================
TOOL REGISTRY: THE ACTION SPACE A FOR PRODUCTION RAG
=============================================================================
The original ReAct paper uses a minimal Wikipedia API:
    Search[entity]:  returns the first 5 sentences of the Wikipedia article.
    Lookup[keyword]: looks up the next occurrence of a keyword in the last paragraph.
    Finish[answer]:  terminates the trajectory with the final answer.

For production RAG we extend this to a richer tool registry:
    search_docs(query):  dense + sparse retrieval from the FAISS+BM25 index.
                         Returns top-5 chunks from the arXiv corpus.
    lookup_term(term):   keyword lookup in the LAST retrieved observation.
                         Simulates Wikipedia's Lookup[] action for passage navigation.
    calculate(expr):     safe eval of arithmetic expressions for numeric reasoning.
    verify_claim(claim): checks whether a claim is supported or refuted by
                         the current accumulated evidence (self-consistency check).
    finish(answer):      terminates the ReAct loop and returns the final answer.

Each tool is a first-class function registered in the TOOL_REGISTRY dict.
Adding a new tool (e.g., SQL query, API call) requires:
    1. Implement the function with (input_str, ctx) -> str signature.
    2. Register: TOOL_REGISTRY["new_tool"] = new_tool_function
    3. Add a description to P_SYSTEM_PROMPT.
That is the complete integration -- no pipeline changes required.

=============================================================================
FEW-SHOT TRAJECTORIES: THE KEY PROMPTING COMPONENT
=============================================================================
ReAct relies on FEW-SHOT PROMPTING with hand-annotated trajectories.
The paper uses 6 examples from HotPotQA training set, each with:
    - Multi-hop question
    - 3-7 Thought/Action/Observation steps
    - Correct final answer via Finish[answer]

The few-shot examples TEACH the LLM:
    - WHEN to issue a Thought (always before an Action)
    - WHAT to put in a Thought (decompose the question, extract key info, plan next step)
    - WHICH action to take (which tool, with what argument)
    - HOW to handle retrieval failures (search reformulation in the next Thought)
    - WHEN to call Finish (when all needed facts are accumulated)

This pipeline implements two few-shot examples embedded in the system prompt.
For production: annotate 6-10 domain-specific examples using your actual corpus.
Fine-tuning on ReAct-format trajectories further improves performance and enables
smaller models to match prompted larger models (paper ablation finding).

"""

'\nReAct-RAG -- Reason + Act Retrieval-Augmented Generation\n=========================================================\nPrimary Reference : Yao et al., "ReAct: Synergizing Reasoning and Acting\n                   in Language Models"\n                   arXiv: 2210.03629  (ICLR 2023, Princeton + Google Brain)\n                   Project: https://react-lm.github.io/\n\nArchitecture: FIVE configurations covering the complete ReAct-RAG design space:\n    (1) CoT-Only Baseline       -- chain-of-thought reasoning, NO tool calls,\n                                   no external retrieval. Pure parametric.\n    (2) Act-Only Baseline       -- tool calls only, NO explicit reasoning traces.\n                                   Actions without Thoughts.\n    (3) Core ReAct              -- canonical Thought -> Action -> Observation\n                                   loop with a multi-tool registry:\n                                   Search[query], Lookup[term], Calculate[expr],\n                   

In [9]:

import os
import sys
import re
import ast
import math
import time
import logging
import operator
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any, Callable


In [10]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [11]:

# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("react_rag")


In [12]:


# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class ReActRAGConfig:
    """
    Centralized configuration for the ReAct-RAG pipeline.

    MAX_STEPS (8):
        Maximum Thought-Action-Observation cycles per trajectory.
        The original ReAct paper uses up to 7 steps on HotPotQA.
        At 8 steps: handles questions requiring up to 4 retrieval hops
        (each hop takes 2 steps: Thought + Action[search] + Observation
        + Thought for the next hop).
        Each step costs 1 LLM call.
        Early termination fires when the model generates Action: finish[...].

    COT_SC_N_TRAJECTORIES (3):
        Number of independent trajectories sampled in Config 4 (ReAct+CoT-SC).
        Yao et al. (2022) test CoT-SC with 21 chains; for ReAct trajectories
        21 is too expensive (~168 LLM calls). N=3 gives meaningful diversity
        at a practical cost (~27 LLM calls). Majority vote selects the answer.

    SAMPLING_TEMPERATURE (0.7):
        Temperature for trajectory sampling in Config 4 (CoT-SC).
        Must be >0 to generate diverse trajectories for majority voting.
        At 0.7: sufficient diversity in Thought phrasing and Action selection
        without collapsing to noise (T>1.0 degrades action grammar).

    REFLECT_MAX_RETRIES (2):
        Maximum reflection-and-retry cycles in Config 5 (ReAct+Reflection).
        Per Shinn et al. (2023) Reflexion framework: each reflection generates
        a critique of the FAILED trajectory and a corrected strategy for the
        next attempt. At 2 retries: handles most single-error failure modes
        (bad search term, wrong entity disambiguation, arithmetic error).

    LOOKUP_WINDOW_SIZE (200):
        Number of characters around a keyword match returned by lookup_term().
        Matches the Wikipedia API's "next occurrence in last paragraph" behavior.
        A window of 200 chars provides ~50 words of context around the match.

    SAFE_CALC_ALLOWED_OPS:
        Whitelist of Python operators allowed in the calculate() tool.
        Prevents arbitrary code execution while enabling full arithmetic.
        The calculate() tool is the ONLY eval() call in this codebase.
        All other inputs are strings passed to the LLM or the retriever.
    """

    # --- Dataset ----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"
    FAISS_DIR:        str = "./faiss_react_rag"

    # --- BGE Embeddings ---------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Chunking ---------------------------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- Retrieval --------------------------------------------------------
    TOP_K_SEARCH:  int = 5
    TOP_K_HYBRID:  int = 3

    # --- ReAct Loop -------------------------------------------------------
    MAX_STEPS:              int = 8
    FINISH_TOKEN:           str = "finish"
    OBSERVATION_MAX_CHARS:  int = 600  # max chars shown to LLM per observation
    LOOKUP_WINDOW_SIZE:     int = 200

    # --- CoT-SC -----------------------------------------------------------
    COT_SC_N_TRAJECTORIES:  int   = 3
    SAMPLING_TEMPERATURE:   float = 0.7

    # --- ReAct + Reflection -----------------------------------------------
    REFLECT_MAX_RETRIES:    int   = 2
    # --- LLM --------------------------------------------------------------
    LLM_TEMPERATURE:         float = 0.0
    AI_FOUNDRY_PROJECT_ENDPOINT: str = "https://idkopenai.services.ai.azure.com"
    AI_FOUNDRY_DEPLOYMENT_NAME: str = "gpt-4o"
    AI_FOUNDRY_API_VERSION: str = "2024-12-01-preview"
    AI_FOUNDRY_API_KEY: str = ""
    LLM_MAX_TOKENS: int            = 512

    # ======================================================================
    # SYSTEM PROMPT WITH FEW-SHOT TRAJECTORIES
    # ======================================================================
    # The few-shot examples are the CORE of ReAct prompting.
    # They demonstrate:
    #   - The exact Thought/Action/Observation format.
    #   - When to use each tool (search_docs vs lookup_term vs calculate).
    #   - How to handle multi-hop reasoning (chain consecutive searches).
    #   - How to handle arithmetic (use calculate, not mental math).
    #   - How to terminate with finish[answer].
    # Production: annotate 6-10 domain-specific trajectories from your corpus.
    P_SYSTEM_PROMPT: str = """You are a precise research assistant that solves questions step by step using Thought-Action-Observation cycles.

AVAILABLE TOOLS:
  search_docs[query]   -- search the document corpus for passages relevant to the query.
  lookup_term[term]    -- look up a specific term in the last retrieved passages.
  calculate[expr]      -- evaluate a safe arithmetic expression (e.g., "512 / 8").
  verify_claim[claim]  -- verify whether a claim is supported by evidence so far.
  finish[answer]       -- return the final answer and terminate.

FORMAT (strictly follow this pattern for every step):
Thought: <reasoning about the current state and what to do next>
Action: <tool_name>[<input>]
Observation: <result of the action -- provided by the system>

RULES:
1. ALWAYS generate a Thought before every Action.
2. ALWAYS call finish[answer] when you have the complete answer.
3. If search results are not helpful, reformulate the query in your next Thought.
4. Never fabricate information -- only use facts from Observations.
5. For arithmetic, ALWAYS use calculate[expr] rather than computing in your Thought.

--- FEW-SHOT EXAMPLE 1 ---
Question: What is the key dimension d_k in the base Transformer model and how many attention heads does it use?

Thought: I need to find the Transformer base model's d_k and number of attention heads. Let me search for this.
Action: search_docs[Transformer base model attention heads d_k dimensions]
Observation: [Vaswani et al.] "We employ h = 8 parallel attention heads. For each, we use d_k = d_v = d_model/h = 64."

Thought: The observation directly answers both parts. d_k = 64 and h = 8. Let me verify the arithmetic.
Action: calculate[512 / 8]
Observation: 64.0

Thought: Confirmed: d_model=512, h=8, d_k = 512/8 = 64. I have a complete and verified answer.
Action: finish[The base Transformer uses h=8 attention heads with d_k = d_model/h = 512/8 = 64 dimensions per head.]
Observation: Finished.

--- FEW-SHOT EXAMPLE 2 ---
Question: What pre-training objective does BERT use that differs from standard language modeling?

Thought: I need to find BERT's pre-training objectives, specifically how they differ from standard LM.
Action: search_docs[BERT pre-training objective masked language model]
Observation: [BERT paper] "BERT is pre-trained using two tasks: Masked LM (MLM) and Next Sentence Prediction (NSP). MLM randomly masks 15% of tokens and predicts them from context."

Thought: The key differentiator is the MLM objective which uses bidirectional context, unlike standard LM which is unidirectional. Let me look up the specific masking details.
Action: lookup_term[15%]
Observation: "randomly mask 15% of WordPiece tokens. Of these, 80% are replaced with [MASK], 10% with a random token, 10% left unchanged."

Thought: I now have both the objective name and the implementation details. The answer is clear.
Action: finish[BERT uses Masked Language Modeling (MLM): 15% of tokens are randomly masked and predicted using bidirectional context. This differs from standard LM which predicts tokens left-to-right only. BERT also uses Next Sentence Prediction (NSP) as a second objective.]
Observation: Finished.

--- END FEW-SHOT EXAMPLES ---

Now solve the following question using the same Thought/Action/Observation format.
"""

    P_COT_ONLY: str = """You are a precise research assistant.
Answer the following question using step-by-step chain-of-thought reasoning.
Use ONLY your internal knowledge -- do NOT reference external tools or searches.
Show your full reasoning process before giving the final answer.

Question: {question}

Chain-of-thought reasoning:"""

    P_REFLECT: str = """You previously attempted to answer the following question but your trajectory was incomplete or incorrect.

Question: {question}

Failed trajectory:
{trajectory}

Failure reason: {failure_reason}

Generate a REFLECTION that:
1. Identifies exactly what went wrong (wrong search query, wrong entity, etc.).
2. Proposes a corrected strategy for the next attempt.
3. Suggests better search terms or lookup terms to try.

Reflection:"""


In [13]:

# ===========================================================================
# SECTION 2 -- DATA STRUCTURES
# ===========================================================================

@dataclass
class ReActStep:
    """
    One Thought-Action-Observation step in a ReAct trajectory.

    thought:     The model's explicit reasoning trace (Thought: ...).
    action_name: The tool called (search_docs, lookup_term, calculate, etc.).
    action_input:The input to the tool.
    observation: The tool's response shown back to the model (Observation: ...).
    is_finish:   True if this step calls finish[answer].
    latency_ms:  Wall-clock time for this step (LLM call + tool execution).
    """
    step_num:      int
    thought:       str
    action_name:   str
    action_input:  str
    observation:   str
    is_finish:     bool    = False
    llm_ms:        float   = 0.0
    tool_ms:       float   = 0.0

    def to_context_text(self) -> str:
        """Format this step for inclusion in the continuing context."""
        lines = [
            f"Thought: {self.thought}",
            f"Action: {self.action_name}[{self.action_input}]",
            f"Observation: {self.observation}",
        ]
        return "\n".join(lines)

    def to_log_line(self) -> str:
        finish_tag = " [FINISH]" if self.is_finish else ""
        return (f"  Step {self.step_num}{finish_tag}: "
                f"{self.action_name}[{self.action_input[:40]}] "
                f"-> obs={self.observation[:60]}")


@dataclass
class ReActTrajectory:
    """
    A complete ReAct trajectory for one query.

    Contains the ordered sequence of ReActSteps and the final answer.
    A trajectory is SUCCESSFUL if it ends with a finish[] action.
    A trajectory is FAILED if it exhausts MAX_STEPS without finishing.
    """
    question:      str
    steps:         List[ReActStep] = field(default_factory=list)
    final_answer:  str             = ""
    is_successful: bool            = False
    reflection:    str             = ""   # populated in Config 5 on failure
    llm_calls:     int             = 0
    total_ms:      float           = 0.0

    def to_full_text(self) -> str:
        """Format the complete trajectory as a text block."""
        lines = [f"Question: {self.question}"]
        for step in self.steps:
            lines.append(step.to_context_text())
        return "\n\n".join(lines)

    def n_steps(self) -> int:
        return len(self.steps)

    def print_trajectory(self) -> None:
        print(f"\n  Trajectory ({self.n_steps()} steps, "
              f"successful={self.is_successful}):")
        for step in self.steps:
            print(step.to_log_line())


@dataclass
class ReActRAGTrace:
    """Execution trace for one question across all five configurations."""
    question:        str
    strategy:        str
    trajectories:    List[ReActTrajectory] = field(default_factory=list)
    selected:        Optional[ReActTrajectory] = None
    final_answer:    str   = ""
    total_llm_calls: int   = 0
    retrieval_ms:    float = 0.0
    generation_ms:   float = 0.0
    total_ms:        float = 0.0

    def print_trace(self) -> None:
        print(f"\n{'='*74}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:74]}")
        print(f"{'='*74}")
        if self.selected:
            self.selected.print_trajectory()
            if self.selected.reflection:
                print(f"\n  REFLECTION:\n  {self.selected.reflection[:200]}")
        print(f"\n  LLM calls: {self.total_llm_calls} | "
              f"Retrieval: {self.retrieval_ms:.0f}ms | "
              f"Total: {self.total_ms:.0f}ms")
        print(f"\n  FINAL ANSWER:\n  {self.final_answer[:450]}")
        print("=" * 74 + "\n")



In [14]:

# ===========================================================================
# SECTION 3 -- INFRASTRUCTURE
# ===========================================================================

def download_pdfs(config: ReActRAGConfig) -> List[Path]:
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            paths.append(dest)
            continue
        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "ReActRAG/1.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            dest.write_bytes(data)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(f"Cannot download '{url}': {exc}") from exc
    return paths


def load_and_chunk(pdf_paths: List[Path], config: ReActRAGConfig) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""], add_start_index=True,
    )
    all_chunks: List[Document] = []
    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        pages      = PyPDFLoader(str(pdf_path)).load()
        for page in pages:
            page.metadata["paper_name"] = paper_name
            page.metadata["source"]     = pdf_path.name
        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d chunks", pdf_path.name, len(chunks))
        all_chunks.extend(chunks)
    return all_chunks


def build_bge_embeddings(config: ReActRAGConfig) -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_faiss(chunks: List[Document], embeddings: HuggingFaceEmbeddings,
                config: ReActRAGConfig) -> FAISS:
    faiss_file = Path(config.FAISS_DIR) / "index.faiss"
    if faiss_file.exists():
        try:
            vs = FAISS.load_local(config.FAISS_DIR, embeddings,
                                  allow_dangerous_deserialization=True)
            return vs
        except Exception:
            pass
    vs = FAISS.from_documents(chunks, embeddings)
    Path(config.FAISS_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_DIR)
    return vs


def build_bm25(chunks: List[Document]) -> BM25Retriever:
    bm25   = BM25Retriever.from_documents(chunks)
    bm25.k = 5
    return bm25


def build_azure_llm(config: ReActRAGConfig,
                    temperature: float = 0.0) -> AzureChatOpenAI:
    """Connect to Azure OpenAI and return an AzureChatOpenAI instance."""
    return AzureChatOpenAI(
        azure_endpoint=getattr(config, 'AI_FOUNDRY_PROJECT_ENDPOINT', 'https://idkopenai.services.ai.azure.com'),
        azure_deployment=getattr(config, 'AI_FOUNDRY_DEPLOYMENT_NAME', 'gpt-4o'),
        api_version=getattr(config, 'AI_FOUNDRY_API_VERSION', '2024-12-01-preview'),
        api_key=getattr(config, 'AI_FOUNDRY_API_KEY', ''),
        temperature=temperature,
        max_tokens=getattr(config, 'LLM_MAX_TOKENS', 512),
    )


def llm_call(messages: list, llm: AzureChatOpenAI) -> str:
    return llm.invoke(messages).content.strip()



In [15]:

# ===========================================================================
# SECTION 4 -- TOOL REGISTRY
# ===========================================================================

class ToolRegistry:
    """
    The ReAct tool registry: maps action names to executable functions.

    Each tool is a callable with signature:
        (input_str: str, ctx: ToolContext) -> str
    where ToolContext carries the retrieval infrastructure and the
    accumulated observation history for lookup_term().

    Adding a new tool:
        1. Define a function with the signature above.
        2. Call registry.register("tool_name", your_function, "description").
        3. Add a description line to P_SYSTEM_PROMPT in the config.
        That is the complete integration.
    """

    def __init__(self) -> None:
        self._tools:        Dict[str, Callable] = {}
        self._descriptions: Dict[str, str]      = {}

    def register(self, name: str, fn: Callable, description: str = "") -> None:
        self._tools[name]        = fn
        self._descriptions[name] = description

    def call(self, name: str, input_str: str, ctx: Any) -> str:
        """
        Execute a registered tool by name with the given input.

        Args:
            name:      tool name (must match a registered key).
            input_str: the argument inside the brackets, e.g. "Search[arg]" -> "arg".
            ctx:       ToolContext with retrieval infrastructure.

        Returns:
            Observation string to feed back to the ReAct loop.
        """
        tool_name = name.lower().strip()
        if tool_name not in self._tools:
            available = list(self._tools.keys())
            return (f"Error: Unknown tool '{name}'. "
                    f"Available tools: {available}. "
                    f"Please use one of the available tools.")
        try:
            return self._tools[tool_name](input_str, ctx)
        except Exception as exc:
            logger.warning("Tool '%s' raised: %s", name, exc)
            return f"Tool error: {exc}. Please try a different query."

    def list_tools(self) -> List[str]:
        return list(self._tools.keys())


@dataclass
class ToolContext:
    """
    Shared context object passed to every tool call.

    vs:              FAISS vectorstore for search_docs.
    bm25:            BM25Retriever for hybrid search.
    last_passages:   the last observation text from search_docs or lookup_term.
                     Used by lookup_term() to find keywords in the LAST retrieved text.
    evidence_texts:  accumulated list of all observations so far.
                     Used by verify_claim() to check claim against ALL evidence.
    retrieval_ms_acc:accumulated retrieval latency across all tool calls.
    config:          ReActRAGConfig.
    """
    vs:                FAISS
    bm25:              BM25Retriever
    config:            ReActRAGConfig
    last_passages:     str        = ""
    evidence_texts:    List[str]  = field(default_factory=list)
    retrieval_ms_acc:  float      = 0.0


def tool_search_docs(query: str, ctx: ToolContext) -> str:
    """
    search_docs[query] -- hybrid dense + sparse retrieval from the corpus.

    Retrieves TOP_K_SEARCH documents from FAISS (dense) and BM25 (sparse),
    deduplicates by page_content hash, and returns the top-3 results formatted
    as "[PaperName pN] content" strings joined by newlines.

    The observation is truncated to OBSERVATION_MAX_CHARS to keep the
    ReAct context window manageable. Full passages are stored in ctx.last_passages
    for use by the subsequent lookup_term() call.
    """
    t0         = time.perf_counter()
    dense_docs = ctx.vs.similarity_search(query, k=ctx.config.TOP_K_SEARCH)
    sparse_docs= ctx.bm25.invoke(query)
    ret_ms     = (time.perf_counter() - t0) * 1000
    ctx.retrieval_ms_acc += ret_ms

    # Deduplicate by content hash, interleave dense and sparse for diversity
    seen     = set()
    combined = []
    for d, s in zip(dense_docs, sparse_docs):
        for doc in [d, s]:
            h = hash(doc.page_content)
            if h not in seen:
                seen.add(h)
                combined.append(doc)
    combined = combined[:ctx.config.TOP_K_HYBRID]

    if not combined:
        return "No relevant passages found. Try a different search query."

    passages = []
    for doc in combined:
        paper  = doc.metadata.get("paper_name", "Unknown")[:20]
        page   = doc.metadata.get("page", "?")
        text   = doc.page_content.strip()[:ctx.config.OBSERVATION_MAX_CHARS // len(combined)]
        passages.append(f"[{paper} p{page}] {text}")

    result = " | ".join(passages)
    ctx.last_passages = result
    ctx.evidence_texts.append(result)
    return result[:ctx.config.OBSERVATION_MAX_CHARS]


def tool_lookup_term(term: str, ctx: ToolContext) -> str:
    """
    lookup_term[term] -- find a keyword in the last retrieved passages.

    Mimics Wikipedia API's Lookup[] action: finds the first occurrence of
    `term` (case-insensitive) in the last retrieved text and returns a
    window of LOOKUP_WINDOW_SIZE characters around the match.

    This is used when search_docs returns a long passage and the model
    needs to zoom into a specific part without triggering another full search.
    """
    if not ctx.last_passages:
        return "No previous passages to look up. Use search_docs first."

    text      = ctx.last_passages
    term_lower = term.lower()
    idx        = text.lower().find(term_lower)

    if idx < 0:
        return (f"Term '{term}' not found in the last retrieved passages. "
                f"Try search_docs with a different query.")

    start  = max(0, idx - ctx.config.LOOKUP_WINDOW_SIZE // 2)
    end    = min(len(text), idx + ctx.config.LOOKUP_WINDOW_SIZE // 2)
    window = text[start:end]
    return f"...{window}..."


def tool_calculate(expr: str, ctx: ToolContext) -> str:
    """
    calculate[expr] -- safely evaluate an arithmetic expression.

    Supports: +, -, *, /, **, //, %, (, ), integers, floats.
    Whitelist of operators prevents code injection.
    Uses Python's ast module to parse and evaluate safely.

    Examples:
        calculate[512 / 8]       -> "64.0"
        calculate[2**10]         -> "1024"
        calculate[100 * 0.15]    -> "15.0"
        calculate[(768 + 512) / 2] -> "640.0"
    """
    ALLOWED_NODES = {
        ast.Expression, ast.BinOp, ast.UnaryOp, ast.Num, ast.Constant,
        ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod,
        ast.Pow, ast.USub, ast.UAdd,
    }

    def safe_eval(node):
        if type(node) not in ALLOWED_NODES:
            raise ValueError(f"Disallowed operation: {type(node).__name__}")
        if isinstance(node, ast.Expression):
            return safe_eval(node.body)
        if isinstance(node, (ast.Num, ast.Constant)):
            return node.n if hasattr(node, 'n') else node.value
        if isinstance(node, ast.UnaryOp):
            val = safe_eval(node.operand)
            return -val if isinstance(node.op, ast.USub) else val
        if isinstance(node, ast.BinOp):
            l, r = safe_eval(node.left), safe_eval(node.right)
            ops = {ast.Add: operator.add, ast.Sub: operator.sub,
                   ast.Mult: operator.mul, ast.Div: operator.truediv,
                   ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
                   ast.Pow: operator.pow}
            if type(node.op) in ops:
                return ops[type(node.op)](l, r)
        raise ValueError(f"Unsupported: {ast.dump(node)}")

    try:
        tree   = ast.parse(expr.strip(), mode="eval")
        result = safe_eval(tree.body)
        return str(result)
    except Exception as exc:
        return f"Calculation error: {exc}. Check the expression syntax."


def tool_verify_claim(claim: str, ctx: ToolContext) -> str:
    """
    verify_claim[claim] -- check if the claim is supported by accumulated evidence.

    A lightweight consistency check: does the claim's key terms appear in
    any of the evidence texts accumulated so far? Returns SUPPORTED, REFUTED
    (if an explicit contradiction is found), or UNCERTAIN.

    This tool is used in Config 3+ to allow the model to SELF-CHECK facts
    before calling finish[]. Reduces false-confidence errors.

    Note: this is a heuristic lexical check, not a semantic entailment model.
    For production: replace with a trained NLI model (e.g., DeBERTa-v3 on MNLI).
    """
    if not ctx.evidence_texts:
        return "UNCERTAIN: No evidence accumulated yet. Use search_docs first."

    claim_terms = set(claim.lower().split())
    all_evidence = " ".join(ctx.evidence_texts).lower()
    found_terms  = [t for t in claim_terms if len(t) > 3 and t in all_evidence]
    coverage     = len(found_terms) / max(len(claim_terms), 1)

    if coverage >= 0.5:
        return (f"SUPPORTED: Key terms found in evidence "
                f"({len(found_terms)}/{len(claim_terms)} claim terms present).")
    return (f"UNCERTAIN: Only {len(found_terms)}/{len(claim_terms)} claim terms "
            f"found in evidence. Consider additional searches.")


def tool_finish(answer: str, ctx: ToolContext) -> str:
    """
    finish[answer] -- terminate the ReAct trajectory with the final answer.

    This is a sentinel tool: when the ReAct loop parser sees this action,
    it marks the trajectory as complete and returns the answer.
    The tool itself just echoes the answer.
    """
    return f"FINAL ANSWER: {answer}"


def build_tool_registry(
    vs: FAISS, bm25: BM25Retriever, config: ReActRAGConfig,
) -> Tuple[ToolRegistry, ToolContext]:
    """
    Build and return the tool registry and shared context for one query session.
    """
    registry = ToolRegistry()
    registry.register("search_docs",  tool_search_docs,  "search the document corpus")
    registry.register("lookup_term",  tool_lookup_term,  "look up a term in last passages")
    registry.register("calculate",    tool_calculate,    "evaluate arithmetic expressions")
    registry.register("verify_claim", tool_verify_claim, "check claim against evidence")
    registry.register("finish",       tool_finish,       "return the final answer")

    ctx = ToolContext(vs=vs, bm25=bm25, config=config)
    return registry, ctx



In [16]:

# ===========================================================================
# SECTION 5 -- REACT TRAJECTORY PARSER
# ===========================================================================

def parse_react_step(raw_output: str) -> Tuple[str, str, str]:
    """
    Parse one Thought-Action pair from the LLM's raw output.

    Expected format (flexible parsing to handle minor format variations):
        Thought: <text>
        Action: <tool_name>[<input>]

    Returns:
        (thought_text, action_name, action_input)
        Returns ("", "finish", "I could not complete the task.") as fallback.

    Parsing is intentionally lenient: the LLM may omit "Thought:" prefix or
    use slight variations. We handle these gracefully to avoid trajectory failures
    due to minor format deviations.
    """
    text = raw_output.strip()

    # Extract Thought
    thought = ""
    thought_match = re.search(r"Thought\s*[:]\s*(.+?)(?=\nAction\s*[:]|\Z)", text,
                               re.DOTALL | re.IGNORECASE)
    if thought_match:
        thought = thought_match.group(1).strip()

    # Extract Action: tool_name[input] or finish[answer]
    action_match = re.search(
        r"Action\s*[:]\s*(\w+)\s*\[(.+?)\]",
        text, re.DOTALL | re.IGNORECASE
    )
    if action_match:
        action_name  = action_match.group(1).strip().lower()
        action_input = action_match.group(2).strip()
        return thought, action_name, action_input

    # Fallback: if model wrote "finish:" without brackets
    finish_match = re.search(r"finish\s*[:]\s*(.+)", text, re.IGNORECASE)
    if finish_match:
        return thought, "finish", finish_match.group(1).strip()

    # Last resort: if LLM just wrote an answer without format
    if not action_match and len(text) > 20:
        return thought, "finish", text[:300]

    return thought, "finish", "I could not determine the answer from available information."



In [17]:

# ===========================================================================
# SECTION 6 -- REACT LOOP ENGINE
# ===========================================================================

def run_react_loop(
    question: str,
    registry: ToolRegistry,
    ctx: ToolContext,
    llm: AzureChatOpenAI,
    config: ReActRAGConfig,
    prior_reflection: str = "",
) -> ReActTrajectory:
    """
    Execute one complete ReAct Thought-Action-Observation loop.

    Builds the trajectory by repeatedly:
        1. Constructing the full context (system prompt + question + steps so far).
        2. Calling the LLM to generate the next Thought + Action.
        3. Parsing the generated output into (thought, action_name, action_input).
        4. Executing the action via the tool registry.
        5. Appending the step (including the Observation) to the trajectory.
        6. Checking for finish[] to terminate.

    If prior_reflection is provided (from Config 5 retry), it is injected into
    the context as additional guidance BEFORE the trajectory starts.

    Args:
        question:          User question.
        registry:          ToolRegistry with all registered tools.
        ctx:               ToolContext (fresh per query, shared across steps).
        llm:               AzureChatOpenAI for step generation.
        config:            ReActRAGConfig.
        prior_reflection:  Optional reflection from a previous failed attempt.

    Returns:
        ReActTrajectory with all steps and final answer.
    """
    trajectory = ReActTrajectory(question=question)
    t0         = time.perf_counter()

    # Build the initial messages (system + user with few-shot examples)
    system_content = config.P_SYSTEM_PROMPT
    if prior_reflection:
        system_content += (f"\n\nIMPORTANT -- Your previous attempt failed. "
                           f"Reflection on what went wrong and corrected strategy:\n"
                           f"{prior_reflection}\n\nApply this corrected strategy now.\n")

    # Accumulate conversation context as a growing string (more efficient than
    # maintaining a full message list that grows with each step)
    context_accumulator = f"Question: {question}\n"

    for step_num in range(1, config.MAX_STEPS + 1):
        # Build messages: system + accumulated context
        messages = [
            SystemMessage(content=system_content),
            HumanMessage(content=context_accumulator
                         + "\nGenerate the next Thought and Action:"),
        ]

        # LLM call: generate next Thought + Action
        t_llm = time.perf_counter()
        raw_output = llm_call(messages, llm)
        llm_ms     = (time.perf_counter() - t_llm) * 1000
        trajectory.llm_calls += 1

        # Parse the output
        thought, action_name, action_input = parse_react_step(raw_output)
        is_finish = (action_name.lower() == config.FINISH_TOKEN)

        # Execute the tool
        t_tool      = time.perf_counter()
        observation = registry.call(action_name, action_input, ctx)
        tool_ms     = (time.perf_counter() - t_tool) * 1000

        if action_name.lower() == "finish":
            # Extract the actual answer from the finish sentinel
            answer_match = re.search(r"FINAL ANSWER:\s*(.+)", observation, re.DOTALL)
            final_ans    = answer_match.group(1).strip() if answer_match else action_input
            observation  = final_ans  # show clean answer in observation

        # Record the step
        step = ReActStep(
            step_num=step_num, thought=thought,
            action_name=action_name, action_input=action_input,
            observation=observation, is_finish=is_finish,
            llm_ms=llm_ms, tool_ms=tool_ms,
        )
        trajectory.steps.append(step)

        logger.info(
            "  Step %d: %s[%s...] -> obs=%.50s",
            step_num, action_name, action_input[:30], observation,
        )

        # Append to context accumulator for next step
        context_accumulator += (
            f"\nThought: {thought}"
            f"\nAction: {action_name}[{action_input}]"
            f"\nObservation: {observation[:200]}\n"
        )

        # Check for trajectory completion
        if is_finish:
            trajectory.final_answer  = step.observation
            trajectory.is_successful = True
            break

    # If loop exhausted without finish
    if not trajectory.is_successful:
        # Use the last non-empty observation as a fallback answer
        for step in reversed(trajectory.steps):
            if step.observation and len(step.observation) > 20:
                trajectory.final_answer = (
                    "Trajectory did not complete. "
                    f"Best available answer from last observation: {step.observation}"
                )
                break
        if not trajectory.final_answer:
            trajectory.final_answer = "Could not determine answer within step limit."

    trajectory.total_ms = (time.perf_counter() - t0) * 1000
    return trajectory



In [18]:

# ===========================================================================
# SECTION 7 -- FIVE REACT-RAG CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: CoT-Only Baseline (no tool use, no retrieval)
# ---------------------------------------------------------------------------

def run_config1_cot_only(
    question: str, llm: AzureChatOpenAI, config: ReActRAGConfig,
) -> ReActRAGTrace:
    """
    Configuration 1: Chain-of-Thought Only (no actions, no retrieval).

    The model reasons about the question using ONLY its parametric knowledge.
    No tool calls, no external retrieval. This is the REASONING-ONLY baseline.

    Failure modes vs ReAct:
        - Fact hallucination: invents specific numbers, names, dates.
        - No mechanism to recover from wrong internal knowledge.
        - Error propagation: one wrong step in the chain leads to wrong answer.

    Per Yao et al. (2022): CoT is particularly prone to hallucination on
    knowledge-intensive tasks like HotPotQA because the model must generate
    intermediate facts from memory rather than retrieving them.

    LLM calls: 1.
    """
    trace = ReActRAGTrace(question=question, strategy="Config1_CoT_Only_NoRetrieval")
    t0    = time.perf_counter()

    prompt = config.P_COT_ONLY.format(question=question)
    t_gen  = time.perf_counter()
    answer = llm_call([HumanMessage(content=prompt)], llm)
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000

    dummy_traj = ReActTrajectory(question=question, final_answer=answer,
                                 is_successful=True, llm_calls=1)
    trace.trajectories    = [dummy_traj]
    trace.selected        = dummy_traj
    trace.final_answer    = answer
    trace.total_llm_calls = 1
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    return trace

In [19]:


# ---------------------------------------------------------------------------
# CONFIG 2: Act-Only Baseline (actions without reasoning traces)
# ---------------------------------------------------------------------------

def run_config2_act_only(
    question: str, registry: ToolRegistry, ctx: ToolContext,
    llm: AzureChatOpenAI, config: ReActRAGConfig,
) -> ReActRAGTrace:
    """
    Configuration 2: Act-Only (tool calls without explicit Thought traces).

    The model executes actions directly without generating Thought traces.
    This ablates the REASONING component of ReAct to isolate its contribution.

    Implemented as a modified system prompt that instructs the model to
    generate ONLY Action: tool[input] lines, no Thought: prefix.
    The loop continues until finish[] or MAX_STEPS.

    Failure modes vs ReAct (Yao et al. 2022):
        - Cannot synthesize across multiple observations (no reasoning state).
        - Cannot reformulate failed searches (no "what went wrong" reasoning).
        - Cannot perform arithmetic reasoning over retrieved numbers.
        - Gets "lost" in multi-hop queries without explicit tracking.

    The act-only vs ReAct comparison on HotPotQA and FEVER (Yao et al.) confirms:
    ReAct consistently outperforms Act-Only because reasoning traces allow the
    model to UPDATE its action plan based on what was retrieved.

    LLM calls: 1-8 (one per step).
    """
    trace = ReActRAGTrace(question=question, strategy="Config2_ActOnly_NoThoughts")
    t0    = time.perf_counter()

    # Override system prompt to suppress Thought generation
    act_only_system = """You are a research assistant that answers questions using tools.
For each step, output ONLY an Action line in this format:
Action: <tool_name>[<input>]

Available tools: search_docs[query], lookup_term[term], calculate[expr], finish[answer]

Do NOT write Thought: lines. Jump directly to Action.
"""
    context = f"Question: {question}\n"
    traj    = ReActTrajectory(question=question)

    for step_num in range(1, config.MAX_STEPS + 1):
        messages   = [SystemMessage(content=act_only_system),
                      HumanMessage(content=context + "\nNext action:")]
        t_llm = time.perf_counter()
        raw   = llm_call(messages, llm)
        llm_ms = (time.perf_counter() - t_llm) * 1000
        traj.llm_calls += 1

        # Parse Action only (no Thought expected)
        action_match = re.search(r"Action\s*[:]\s*(\w+)\s*\[(.+?)\]", raw,
                                  re.DOTALL | re.IGNORECASE)
        if action_match:
            action_name  = action_match.group(1).strip().lower()
            action_input = action_match.group(2).strip()
        else:
            action_name, action_input = "finish", raw[:200]

        t_tool = time.perf_counter()
        obs    = registry.call(action_name, action_input, ctx)
        tool_ms= (time.perf_counter() - t_tool) * 1000

        is_finish = (action_name == "finish")
        if is_finish:
            answer_match = re.search(r"FINAL ANSWER:\s*(.+)", obs, re.DOTALL)
            obs = answer_match.group(1).strip() if answer_match else action_input

        step = ReActStep(step_num=step_num, thought="",  # no thought in Act-Only
                         action_name=action_name, action_input=action_input,
                         observation=obs, is_finish=is_finish,
                         llm_ms=llm_ms, tool_ms=tool_ms)
        traj.steps.append(step)
        context += f"\nAction: {action_name}[{action_input}]\nObservation: {obs[:200]}\n"

        if is_finish:
            traj.final_answer  = obs
            traj.is_successful = True
            break

    if not traj.is_successful:
        for s in reversed(traj.steps):
            if s.observation and len(s.observation) > 10:
                traj.final_answer = s.observation
                break

    trace.trajectories    = [traj]
    trace.selected        = traj
    trace.final_answer    = traj.final_answer
    trace.total_llm_calls = traj.llm_calls
    trace.retrieval_ms    = ctx.retrieval_ms_acc
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    return trace


In [20]:

# ---------------------------------------------------------------------------
# CONFIG 3: Core ReAct (Thought + Action + Observation interleaved)
# ---------------------------------------------------------------------------

def run_config3_core_react(
    question: str, registry: ToolRegistry, ctx: ToolContext,
    llm: AzureChatOpenAI, config: ReActRAGConfig,
) -> ReActRAGTrace:
    """
    Configuration 3: Core ReAct -- canonical Thought-Action-Observation loop.

    This is the central contribution of Yao et al. (2022): interleaving
    explicit reasoning traces (Thoughts) with tool-augmented actions.

    The synergy between Thought and Action:
        Thoughts help the model:
            - Decompose multi-hop questions into sub-problems.
            - Target which specific information to retrieve next.
            - Track progress toward the final answer.
            - Handle retrieval failures by reformulating.
        Actions (tool calls) help the model:
            - Retrieve factual information from external sources.
            - Ground reasoning in verifiable external evidence.
            - Perform exact arithmetic rather than approximating.
        Observations update the model's reasoning state:
            - New facts from observations trigger new Thoughts.
            - Empty observations trigger query reformulation Thoughts.

    This config uses the full multi-tool registry:
        search_docs, lookup_term, calculate, verify_claim, finish.
    The model selects which tool to use based on the current Thought.

    LLM calls: 1 per step, typically 3-6 for multi-hop questions.
    """
    trace = ReActRAGTrace(question=question, strategy="Config3_Core_ReAct")
    t0    = time.perf_counter()

    traj = run_react_loop(question, registry, ctx, llm, config)
    trace.trajectories    = [traj]
    trace.selected        = traj
    trace.final_answer    = traj.final_answer
    trace.total_llm_calls = traj.llm_calls
    trace.retrieval_ms    = ctx.retrieval_ms_acc
    trace.generation_ms   = sum(s.llm_ms for s in traj.steps)
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    return trace



In [21]:

# ---------------------------------------------------------------------------
# CONFIG 4: ReAct + CoT-SC (Self-Consistency sampling)
# ---------------------------------------------------------------------------

def run_config4_react_cot_sc(
    question: str, vs: FAISS, bm25: BM25Retriever,
    llm_deterministic: AzureChatOpenAI, config: ReActRAGConfig,
) -> ReActRAGTrace:
    """
    Configuration 4: ReAct + Chain-of-Thought Self-Consistency.

    Samples N=COT_SC_N_TRAJECTORIES independent ReAct trajectories with
    temperature=SAMPLING_TEMPERATURE (stochastic) and selects the final answer
    by MAJORITY VOTE.

    Rationale: A single ReAct trajectory may reach the wrong answer due to:
        - Unlucky search results for a sub-optimal query formulation.
        - Reasoning errors in one Thought that propagate.
        - Ambiguous observations that the model misinterprets.

    With N=3 trajectories: each uses a DIFFERENT sequence of Thoughts and
    Actions (different search queries, different reasoning paths) due to
    temperature-induced diversity. The answer that appears in the majority
    of trajectories is more likely to be correct (Wang et al., 2022 CoT-SC).

    Per Yao et al. (2022): combining ReAct with CoT-SC achieves SOTA performance
    on HotPotQA and FEVER. The combination uses ReAct for grounding and CoT-SC
    for uncertainty reduction.

    Majority vote: extracts the "core answer" from each finish[] call,
    groups trajectories by their answer, and returns the most common answer.

    LLM calls: N * MAX_STEPS (worst case). Typical: N * 5 = 15.
    """
    trace = ReActRAGTrace(question=question, strategy="Config4_ReAct_CoTSC")
    t0    = time.perf_counter()

    # Build a sampling LLM (T=0.7)
    llm_sampling = build_azure_llm(config, temperature=config.SAMPLING_TEMPERATURE)

    trajectories: List[ReActTrajectory] = []
    total_llm_calls = 0
    total_ret_ms    = 0.0

    logger.info("Config4 ReAct+CoT-SC: sampling %d trajectories (T=%.1f) ...",
                config.COT_SC_N_TRAJECTORIES, config.SAMPLING_TEMPERATURE)

    for n in range(config.COT_SC_N_TRAJECTORIES):
        # Fresh ToolContext per trajectory (independent retrieval state)
        _, fresh_ctx = build_tool_registry(vs, bm25, config)
        registry, _ = build_tool_registry(vs, bm25, config)

        logger.info("  Trajectory %d/%d ...", n + 1, config.COT_SC_N_TRAJECTORIES)
        traj = run_react_loop(
            question, registry, fresh_ctx, llm_sampling, config
        )
        trajectories.append(traj)
        total_llm_calls += traj.llm_calls
        total_ret_ms    += fresh_ctx.retrieval_ms_acc

        logger.info(
            "  Trajectory %d: steps=%d, success=%s, answer=%.60s",
            n + 1, traj.n_steps(), traj.is_successful, traj.final_answer,
        )

    # Majority vote: normalize answers, count occurrences
    def normalize_answer(ans: str) -> str:
        return re.sub(r"\s+", " ", ans.strip().lower())[:100]

    answer_counts: Dict[str, int]              = {}
    answer_to_traj: Dict[str, ReActTrajectory] = {}
    for traj in trajectories:
        norm = normalize_answer(traj.final_answer)
        answer_counts[norm] = answer_counts.get(norm, 0) + 1
        if norm not in answer_to_traj:
            answer_to_traj[norm] = traj

    best_norm   = max(answer_counts, key=lambda k: answer_counts[k])
    best_traj   = answer_to_traj[best_norm]
    best_count  = answer_counts[best_norm]
    logger.info(
        "Config4 CoT-SC: selected answer with %d/%d votes (normalized='%.60s')",
        best_count, config.COT_SC_N_TRAJECTORIES, best_norm,
    )

    trace.trajectories    = trajectories
    trace.selected        = best_traj
    trace.final_answer    = best_traj.final_answer
    trace.total_llm_calls = total_llm_calls
    trace.retrieval_ms    = total_ret_ms
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    return trace



In [22]:


# ---------------------------------------------------------------------------
# CONFIG 5: ReAct + Reflection [BEST]
# ---------------------------------------------------------------------------

def run_config5_react_reflection(
    question: str, registry: ToolRegistry, ctx: ToolContext,
    llm: AzureChatOpenAI, config: ReActRAGConfig,
) -> ReActRAGTrace:
    """
    Configuration 5: ReAct + Reflection (Reflexion-style self-correction) [BEST].

    Combines the core ReAct loop (Config 3) with ACTIVE FAILURE DETECTION and
    SELF-REFLECTION from Shinn et al. (2023) "Reflexion: Language Agents with
    Verbal Reinforcement Learning."

    Mechanism:
        Attempt 1: Run a standard ReAct trajectory.
        If the trajectory FAILS (is_successful=False -- exhausted MAX_STEPS
        without finishing) or returns an obviously incomplete answer:
            - Generate a REFLECTION: an explicit critique of what went wrong
              and a corrected strategy for the next attempt.
            - Attempt 2: Run a new ReAct trajectory with the reflection injected
              into the system prompt as additional guidance.
            - Repeat for REFLECT_MAX_RETRIES attempts.

    The reflection is generated by prompting the LLM with the FULL failed
    trajectory and asking it to identify the failure mode and propose corrections.
    This is "verbal reinforcement learning": instead of updating model weights
    via gradient descent, the model updates its STRATEGY via natural language
    self-critique.

    Common failure modes addressed by reflection:
        - Search query too broad: "search_docs[attention]" -> reflect:
          "My query was too broad. Next attempt: use specific paper name and
           parameter name: 'Transformer attention heads d_k base model'"
        - Entity disambiguation: retrieved wrong "BERT" article -> reflect:
          "I retrieved BERT (TV show) instead of BERT (ML paper). Next attempt:
           include 'NLP pre-training' in the search to disambiguate."
        - Missing arithmetic verification: stated wrong calculation -> reflect:
          "I calculated in my Thought instead of using calculate[]. Always use
           the calculate tool for numeric operations."

    This config achieves the highest answer quality by combining:
        - ReAct's grounding (actions prevent hallucination)
        - Reflection's error recovery (failure triggers self-improvement)
        - Full trajectory audit (every step logged for interpretability)

    LLM calls: (MAX_STEPS per attempt) * (1 + REFLECT_MAX_RETRIES) worst case.
               Typical: 5-12 calls for a successful 2-attempt trajectory.
    """
    trace = ReActRAGTrace(
        question=question,
        strategy="Config5_ReAct_Reflection [BEST]",
    )
    t0 = time.perf_counter()

    all_trajectories:  List[ReActTrajectory] = []
    total_llm_calls    = 0
    total_ret_ms       = 0.0
    prior_reflection   = ""

    for attempt in range(1 + config.REFLECT_MAX_RETRIES):
        logger.info(
            "Config5 ReAct+Reflection: attempt %d/%d ...",
            attempt + 1, 1 + config.REFLECT_MAX_RETRIES,
        )

        # Fresh ToolContext per attempt (independent retrieval state)
        registry_fresh, ctx_fresh = build_tool_registry(ctx.vs, ctx.bm25, config)

        traj = run_react_loop(
            question, registry_fresh, ctx_fresh, llm, config,
            prior_reflection=prior_reflection,
        )
        all_trajectories.append(traj)
        total_llm_calls += traj.llm_calls
        total_ret_ms    += ctx_fresh.retrieval_ms_acc

        logger.info(
            "  Attempt %d: steps=%d, success=%s, answer=%.60s",
            attempt + 1, traj.n_steps(), traj.is_successful, traj.final_answer,
        )

        # If successful: no reflection needed
        if traj.is_successful and len(traj.final_answer) > 20:
            break

        # If not the last attempt: generate reflection
        if attempt < config.REFLECT_MAX_RETRIES:
            failure_reason = (
                "Trajectory exhausted max steps without calling finish[]."
                if not traj.is_successful
                else "Trajectory finished but answer appears incomplete or incorrect."
            )
            reflect_prompt = config.P_REFLECT.format(
                question=question,
                trajectory=traj.to_full_text()[:1500],
                failure_reason=failure_reason,
            )
            t_reflect = time.perf_counter()
            prior_reflection = llm_call([HumanMessage(content=reflect_prompt)], llm)
            total_llm_calls += 1

            traj.reflection = prior_reflection
            logger.info(
                "  Reflection generated: %.80s ...",
                prior_reflection[:80],
            )

    # Select the best trajectory (last successful, or last attempted)
    best_traj = None
    for traj in reversed(all_trajectories):
        if traj.is_successful:
            best_traj = traj
            break
    if best_traj is None:
        best_traj = all_trajectories[-1]

    trace.trajectories    = all_trajectories
    trace.selected        = best_traj
    trace.final_answer    = best_traj.final_answer
    trace.total_llm_calls = total_llm_calls
    trace.retrieval_ms    = total_ret_ms
    trace.generation_ms   = sum(sum(s.llm_ms for s in t.steps)
                                for t in all_trajectories)
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    return trace



In [23]:

# ===========================================================================
# SECTION 8 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question: str, vs: FAISS, bm25: BM25Retriever,
    llm: AzureChatOpenAI, config: ReActRAGConfig,
) -> Dict[str, ReActRAGTrace]:
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    def make_registry_and_ctx():
        return build_tool_registry(vs, bm25, config)

    runners = {
        "Config1_CoT_Only":              lambda q: run_config1_cot_only(
            q, llm, config),
        "Config2_Act_Only":              lambda q: run_config2_act_only(
            q, *make_registry_and_ctx(), llm, config),
        "Config3_Core_ReAct":            lambda q: run_config3_core_react(
            q, *make_registry_and_ctx(), llm, config),
        "Config4_ReAct_CoTSC":           lambda q: run_config4_react_cot_sc(
            q, vs, bm25, llm, config),
        "Config5_ReAct_Reflection [BEST]":lambda q: run_config5_react_reflection(
            q, *make_registry_and_ctx(), llm, config),
    }

    results: Dict[str, ReActRAGTrace] = {}
    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            trace = fn(question)
            trace.print_trace()
            results[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    print("\n" + "=" * 78)
    print("REACT-RAG COMPARISON SUMMARY")
    print(f"{'Config':<44} {'Trajs':>5} {'Steps':>5} {'LLM':>5} "
          f"{'Ret_ms':>7} {'Total_ms':>9}")
    print("-" * 78)
    for lbl, tr in results.items():
        sel  = tr.selected
        stps = sel.n_steps() if sel else 0
        trajs = len(tr.trajectories)
        print(f"{lbl:<44} {trajs:>5} {stps:>5} {tr.total_llm_calls:>5} "
              f"{tr.retrieval_ms:>7.0f} {tr.total_ms:>9.0f}")
    print("=" * 78 + "\n")
    return results




In [24]:
"""
    End-to-end ReAct-RAG pipeline orchestrator.

    INDEXING (run once, cached): ~2 min.
        - Download 3 arXiv PDFs.
        - Chunk -> BGE-large embed -> FAISS index.
        - Build BM25 index in memory.

    QUERY COSTS PER QUESTION:
        Config 1 (CoT-Only):       1 LLM call.
        Config 2 (Act-Only):       1-8 LLM calls.
        Config 3 (Core ReAct):     1-8 LLM calls (typically 4-6).
        Config 4 (ReAct+CoT-SC):   N=3 trajectories * 4-6 steps = 12-18 calls.
        Config 5 (ReAct+Reflect):  1-2 attempts * 4-6 steps + 1 reflect = 9-13 calls.

    DEMO QUESTIONS: designed to stress-test each config's strengths/weaknesses.
        - Multi-hop: requires bridging across two facts (CoT fails).
        - Numeric: requires arithmetic (Act-Only fails without calculate[]).
        - Ambiguous: requires search reformulation (single-trajectory ReAct struggles).
    """

"\n    End-to-end ReAct-RAG pipeline orchestrator.\n\n    INDEXING (run once, cached): ~2 min.\n        - Download 3 arXiv PDFs.\n        - Chunk -> BGE-large embed -> FAISS index.\n        - Build BM25 index in memory.\n\n    QUERY COSTS PER QUESTION:\n        Config 1 (CoT-Only):       1 LLM call.\n        Config 2 (Act-Only):       1-8 LLM calls.\n        Config 3 (Core ReAct):     1-8 LLM calls (typically 4-6).\n        Config 4 (ReAct+CoT-SC):   N=3 trajectories * 4-6 steps = 12-18 calls.\n        Config 5 (ReAct+Reflect):  1-2 attempts * 4-6 steps + 1 reflect = 9-13 calls.\n\n    DEMO QUESTIONS: designed to stress-test each config's strengths/weaknesses.\n        - Multi-hop: requires bridging across two facts (CoT fails).\n        - Numeric: requires arithmetic (Act-Only fails without calculate[]).\n        - Ambiguous: requires search reformulation (single-trajectory ReAct struggles).\n    "

In [25]:
config = ReActRAGConfig()


In [26]:
pdf_paths  = download_pdfs(config)
chunks     = load_and_chunk(pdf_paths, config)
embeddings = build_bge_embeddings(config)
vs         = build_faiss(chunks, embeddings, config)
bm25       = build_bm25(chunks)
llm        = build_azure_llm(config, temperature=0.0)


2026-05-27 22:25:43 | INFO     | react_rag |   attention_is_all_you_need.pdf -> 104 chunks
2026-05-27 22:25:43 | INFO     | react_rag |   bert_pretraining_transformers.pdf -> 173 chunks
2026-05-27 22:25:44 | INFO     | react_rag |   rag_knowledge_intensive_nlp.pdf -> 181 chunks
2026-05-27 22:25:44 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


C:\Users\dhanu\AppData\Local\Temp\ipykernel_15968\466755378.py:46: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


2026-05-27 22:25:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-05-27 22:25:44 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-27 22:25:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-27 22:25:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:25:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-27 22:25:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5931.12it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-27 22:25:47 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:25:47 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-27 22:25:48 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:25:48 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-27 22:25:48 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-27 22:25:48 | INFO     | httpx |

In [27]:
demo_questions = [
    # Multi-hop: two retrieval steps needed (Transformer params -> calculation)
    "How many total parameters are in the attention heads of the Transformer base model's encoder, and how was this calculated?",

    # Numeric with arithmetic: verify specific calculation from the paper
    "What is d_model divided by the number of attention heads h in the base Transformer, and what does this value represent?",

    # Cross-paper multi-hop: BERT vs Transformer architecture question
    "What pre-training technique does BERT use that the original Transformer architecture does not include, and what bidirectional property does it enable?",

    # Claim verification: tests the verify_claim tool
    "Is it correct that BERT uses 12 transformer encoder layers in its base configuration, and what is the model's hidden size?",

    # Complex decomposition: needs multiple separate lookups
    "What are the three main components of a Transformer encoder block, and what is the purpose of the feed-forward sub-layer's inner dimension?",
]


In [28]:
for question in demo_questions[:2]:   # run 2 for demo; all 5 is expensive
    run_all_configs(question, vs, bm25, llm, config)

logger.info("=== ReAct-RAG Pipeline Demo Complete ===")



##############################################################################
QUERY: How many total parameters are in the attention heads of the Transformer base model's encoder, and how was this calculated?
##############################################################################

RUNNING: Config1_CoT_Only
2026-05-27 22:26:05 | INFO     | httpx | HTTP Request: POST https://idkopenai.services.ai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"

TRACE: Config1_CoT_Only_NoRetrieval
Query: How many total parameters are in the attention heads of the Transformer ba

  Trajectory (0 steps, successful=True):

  LLM calls: 1 | Retrieval: 0ms | Total: 8354ms

  FINAL ANSWER:
  To calculate the total number of parameters in the attention heads of the Transformer base model's encoder, we need to carefully analyze the architecture of the attention mechanism and its parameterization. Here's the step-by-step reasoning:

---

### Step 1: Und

In [4]:
# Extra ReAct test questions (multi-hop, arithmetic, and verification)
extra_demo_questions = [
    "What is d_model/h in Transformer base, and why does this value matter for each attention head?",
    "BERT-Base has how many encoder layers and what hidden size? Verify each claim with retrieved evidence.",
    "How does RAG combine parametric and non-parametric memory during answer generation?",
]

RUN_EXTRA_DEMOS = False
if RUN_EXTRA_DEMOS:
    for question in extra_demo_questions[:2]:
        run_all_configs(question, vs, bm25, llm, config)
else:
    print("Set RUN_EXTRA_DEMOS = True to execute extra ReAct examples.")

Set RUN_EXTRA_DEMOS = True to execute extra ReAct examples.
